# RF Jamming / GNSS Interference / CRPA — tech-intel pipeline (JNWC)

Clone of the `techintel-quantum` / `hypersonic` OpenAlex pipeline, re-run from scratch on a
**keyword Works corpus** for navigation-warfare topics. Produces the three `.pkl.gz` frames
plus the two lookup dicts consumed by `jamming2d.py`.

In [1]:
from pyalex import (
    Works, Authors, Sources,
    Institutions, Concepts, Publishers, Funders
)
import pyalex
import pandas as pd
import numpy as np
pyalex.config.email = "doolingdavid@gmail.com"

from flair.embeddings import DocumentPoolEmbeddings
from flair.data import Sentence
from flair.embeddings import SentenceTransformerDocumentEmbeddings

EMBEDDING_MODEL_1 = "all-mpnet-base-v2"

# this one is also good: all-MiniLM-L6-v2
EMBEDDING_MODEL_2 = "all-MiniLM-L6-v2"
SENT_EMBEDDINGS_1 = SentenceTransformerDocumentEmbeddings(EMBEDDING_MODEL_1)
SENT_EMBEDDINGS_2 = SentenceTransformerDocumentEmbeddings(EMBEDDING_MODEL_2)
DOC_EMBEDDINGS = DocumentPoolEmbeddings([SENT_EMBEDDINGS_2])

import torch
from tqdm import tqdm
import yake
import umap.umap_ as umap
import textwrap

/home/david/jobs/openalex_local/techintel-jamming/.venv/lib/python3.12/site-packages/flair/embeddings/document.py:591: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return self.model.get_sentence_embedding_dimension()


## 1. Corpus — exact-phrase OpenAlex `Works` search (title + abstract)

CRPA / anti-jam antenna hardware is not an OpenAlex *concept*, so we pull the corpus by
**exact-phrase** search (`title_and_abstract.search` with quoted phrases) over a curated
navigation-warfare phrase list, then union + dedupe by paper id. Exact-phrase + title/abstract
(not full text) keeps the corpus on-target — e.g. "navigation warfare" matches the phrase, not
papers that merely contain *navigation* and *warfare* somewhere.

In [2]:
SEARCH_PHRASES = [
    # jamming / interference
    "GPS jamming",
    "GNSS jamming",
    "GNSS jamming detection",
    "GNSS interference",
    "GPS interference",
    "GNSS interference mitigation",
    "jammer localization",
    # spoofing
    "GNSS spoofing",
    "GPS spoofing",
    "GNSS spoofing detection",
    "GPS spoofing detection",
    # CRPA / anti-jam antenna hardware
    "controlled reception pattern antenna",
    "controlled reception pattern",
    "anti-jam antenna",
    "anti-jamming antenna",
    "null steering antenna",
    "adaptive antenna nulling",
    "adaptive beamforming antenna",
    "GNSS anti-jamming",
    "GPS anti-jamming",
    "anti-jam GPS",
    # mission framing
    "navigation warfare",
    "PNT resilience",
    "resilient PNT",
]
# safety cap per phrase (no single exact phrase approaches this; keeps the pull bounded)
N_MAX_PER_PHRASE = 2000

In [3]:
def process_works_list(worklist:list):
    """
    transforms the
    works list into a dataframe.
    """
    abstracts_dict = {h["id"]:h["abstract"] for h in worklist}
    df = pd.DataFrame.from_records(worklist)
    df.drop(columns=['abstract_inverted_index'], errors='ignore', inplace=True)
    df['abstract'] = df['id'].map(abstracts_dict)
    return df

In [4]:
def get_search_frame(phrase:str, n_max:int=N_MAX_PER_PHRASE):
    """
    exact-phrase Works search for `phrase` over title+abstract,
    paginated and deduped by paper id, mirroring get_concept_frame.
    """
    pager = Works().filter(
        publication_year='>2000',
        title_and_abstract={"search": f'"{phrase}"'},
    ).paginate(per_page=200, n_max=n_max)
    df = pd.DataFrame()
    for page in tqdm(pager, desc=phrase):
        if not page:
            continue
        dfpage = process_works_list(page)
        df = pd.concat([df, dfpage], ignore_index=True)
        df.drop_duplicates(subset='id', keep='first', inplace=True)
    return df

In [5]:
frames_list = []
for phrase in SEARCH_PHRASES:
    df = get_search_frame(phrase)
    print(f"{phrase!r}: {len(df)} papers")
    frames_list.append(df)
len(frames_list)

GPS jamming: 0it [00:00, ?it/s]

GPS jamming: 1it [00:01,  1.15s/it]

GPS jamming: 2it [00:03,  1.66s/it]

GPS jamming: 2it [00:03,  1.58s/it]

'GPS jamming': 375 papers


...

resilient PNT: 0it [00:00, ?it/s]

resilient PNT: 1it [00:08,  8.08s/it]

resilient PNT: 1it [00:08,  8.08s/it]

'resilient PNT': 82 papers


24

In [6]:
dftop = pd.concat(frames_list,
                  ignore_index=True)
dftop.drop_duplicates(subset='id', keep='first',
                      inplace=True)

dftop.set_index('id', inplace=True, drop=False)

dfall = dftop
print("corpus shape:", dfall.shape)

dfall['content'] = dfall['title'] + ". " + dfall['abstract']

dfrecords = dfall[~dfall['content'].isna()].copy()
print("with content:", dfrecords.shape)

# OpenAlex no longer always returns a 'grants' field (and 'locations' can be
# absent for sparse records); guarantee the columns the pipeline selects on
# exist, defaulting to an empty list per row.
for _col in ['grants', 'locations']:
    if _col not in dfrecords.columns:
        dfrecords[_col] = [[] for _ in range(len(dfrecords))]
    else:
        dfrecords[_col] = dfrecords[_col].apply(
            lambda v: v if isinstance(v, list) else [])

corpus shape: (3486, 50)
with content: (3018, 51)


## 2. Keywords + top concepts

In [7]:
def get_keywords(text:str, top:int=7, stopwords=None):
    """
    takes a blob of text and
    returns the top **top**
    keywords as a list
    """
    kw_extractor = yake.KeywordExtractor(top=top, stopwords=stopwords)
    keywords = kw_extractor.extract_keywords(text)
    return [p[0] for p in keywords]


def get_top_concepts(concept_list:list, score:float=.6):
    """
    takes a list of concept dictionaries
    returns the display_names of concepts whose score is >= score
    """
    return [c['display_name'] for c in concept_list if c['score'] >= score]

In [8]:
dfrecords['keywords'] = dfrecords['content'].apply(get_keywords)
dfrecords['top_concepts'] = dfrecords['concepts'].apply(get_top_concepts)

## 3. Sentence embeddings → UMAP (2D) → HDBSCAN clusters

In [9]:
def get_content_embeddings(dfrecords:pd.DataFrame) -> pd.DataFrame:
    """
    passes the preprocessed content strings through the embedding model
    to produce the vector-space representation of each paper.
    """
    sent = Sentence("The grass is green.")
    DOC_EMBEDDINGS.embed(sent)
    texts = dfrecords["content"].str.lower().values.tolist()
    all_descriptions = np.empty((len(texts), len(sent.embedding)))
    for i in tqdm(range(len(texts))):
        sent = Sentence(texts[i])
        DOC_EMBEDDINGS.embed(sent)
        all_descriptions[i, :] = sent.embedding.cpu().numpy()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    dfcontentvectors = pd.DataFrame.from_records(all_descriptions, index=dfrecords.index)
    return dfcontentvectors

In [10]:
dfcontentvectors = get_content_embeddings(dfrecords)

  0%|          | 0/3018 [00:00<?, ?it/s]

  0%|          | 2/3018 [00:00<03:02, 16.53it/s]

  0%|          | 5/3018 [00:00<02:29, 20.14it/s]

  0%|          | 7/3018 [00:00<02:33, 19.56it/s]

  0%|          | 10/3018 [00:00<02:13, 22.59it/s]

...

 91%|█████████ | 2748/3018 [02:37<00:14, 19.22it/s]

 95%|█████████▍| 2864/3018 [02:42<00:07, 20.83it/s]

 95%|█████████▍| 2867/3018 [02:42<00:07, 20.92it/s]

 99%|█████████▉| 2983/3018 [02:47<00:01, 22.18it/s]

100%|██████████| 3018/3018 [02:41<00:00, 18.66it/s]

In [11]:
N_COMPONENTS = 2  # can visualize this way
umap_reducer = umap.UMAP(n_components=N_COMPONENTS,
                         random_state=1234,
                         metric='cosine')
# keep this fitted reducer to transform the topic-centroid mean embeddings later.

reduced_vectors = umap_reducer.fit_transform(dfcontentvectors.to_numpy())
dfreduced = pd.DataFrame.from_records(reduced_vectors,
                index=dfcontentvectors.index)
dfreduced.columns = ['x','y']

/home/david/jobs/openalex_local/techintel-jamming/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [12]:
import hdbscan

hdbscan_args = {'min_cluster_size': 15,
                'metric': 'euclidean',
                'cluster_selection_method': 'eom',
                'cluster_selection_epsilon': 0.1}

cluster = hdbscan.HDBSCAN(**hdbscan_args).fit(dfreduced[['x','y']].to_numpy())

dfreduced['cluster'] = cluster.labels_
dfreduced['probability'] = cluster.probabilities_

dfpapers = dfrecords.merge(dfreduced, left_index=True,
                           right_index=True)
print("clusters (excl noise):", (dfreduced['cluster'].nunique() - (1 if -1 in dfreduced['cluster'].values else 0)))

clusters (excl noise): 25


## 4. Explode authorships → paper/author/affiliation triples

In [13]:
del dfpapers['id']
dfstart = dfpapers.reset_index()
dfstart.shape

(3018, 57)

In [14]:
dfbig = dfstart.explode(column='authorships')
dfbig.shape, dfstart.shape

((10593, 57), (3018, 57))

In [15]:
def add_extra_to_authorships(row: pd.DataFrame):
    """
    row[authorships] is a dictionary;
    add in the paper-level keys to that dictionary.
    """
    complete_dict = row["authorships"]
    if type(complete_dict) == dict:
        complete_dict["id"] = row["id"]
        complete_dict["x"] = row["x"]
        complete_dict["y"] = row["y"]
        complete_dict["cluster"] = row["cluster"]
        complete_dict["cluster_score"] = row["probability"]
        complete_dict["title"] = row["title"]
        complete_dict["abstract"] = row["abstract"]
        complete_dict["doi"] = row["doi"]
        complete_dict["publication_date"] = row["publication_date"]
        complete_dict["publication_year"] = row["publication_year"]
        complete_dict["grants"] = row["grants"]
        complete_dict["locations"] = row["locations"]
        return complete_dict
    else:
        return row["authorships"]

In [16]:
dfbig['big_authorships'] = dfbig.apply(add_extra_to_authorships, axis=1)

In [17]:
bigvals = dfbig['authorships'].tolist()
dictvals = [c for c in bigvals if type(c) != float]

In [18]:
dftriple = pd.json_normalize(dictvals,
                  record_path=['institutions'],
                  meta=['id','raw_affiliation_string','author_position', 'doi',
                        'title','abstract','publication_date', 'publication_year',
                        'grants','locations',
                        'is_corrresponding','x','y','cluster','cluster_score',
                       ['author','id'], ['author', 'display_name'],
                       ['author','orcid']],
                  errors='ignore',
                  sep='_',
                  meta_prefix='paper_',
                 )
dftriple.shape

(7748, 24)

## 5. Topic centroids (keywords + concepts per cluster)

In [19]:
dftopics = dfcontentvectors.copy()
dftopics['cluster'] = dfpapers['cluster']
dfmeantopics = dftopics.groupby('cluster').mean().copy()
reduced_topics = umap_reducer.transform(dfmeantopics.to_numpy())
df_reduced_topics = pd.DataFrame.from_records(reduced_topics,
                index=dfmeantopics.index)
df_reduced_topics.columns = ['x','y']
df_reduced_topics['topic'] = df_reduced_topics.index


def get_cluster_concepts(topic_num:int, n:int=20):
    """
    returns the list of top n occurring concepts from the top_concepts field
    for the given cluster.
    """
    top_concepts = dfpapers[dfpapers['cluster'] == topic_num]['top_concepts'].tolist()
    flat_concepts = [item for sublist in top_concepts for item in sublist]
    concepts_dict = {c:flat_concepts.count(c) for c in flat_concepts}
    sorted_concepts = sorted(concepts_dict.items(), key=lambda x:x[1], reverse=True)
    return [c[0] for c in sorted_concepts][:n]


def get_yake_cluster_phrases(topic_num:int, n:int=20):
    """
    returns the list of YAKE keyphrases for the concatenated abstracts of a cluster.
    """
    documents = dfpapers[dfpapers['cluster'] == topic_num]['content'].tolist()
    topic_input = ". ".join(documents)
    kw_extractor = yake.KeywordExtractor(top=n, stopwords=None)
    keywords = kw_extractor.extract_keywords(topic_input)
    return [p[0] for p in keywords]


wikiconcepts = df_reduced_topics['topic'].apply(get_cluster_concepts)
wikikeywords = df_reduced_topics['topic'].apply(get_yake_cluster_phrases)

dfpapers['id'] = dfpapers.index
dfinfo = dfpapers[['x','y','id','title','doi','cluster','grants',
                   'locations',
                 'publication_date','keywords','top_concepts']].copy()

centroids = dfinfo.groupby('cluster')[['x','y']].mean().copy()
centroids['concepts'] = wikiconcepts
centroids['cluster'] = centroids.index
centroids['keywords'] = wikikeywords

In [20]:
def wrap_it(x):
    return "<br>".join(textwrap.wrap(x, width=40))

centroids['wrapped_keywords'] = centroids['keywords'].apply(str).apply(wrap_it)
centroids['wrapped_concepts'] = centroids['concepts'].apply(str).apply(wrap_it)

## 6. Enrich dfinfo and dftriple (source / funders / affil & author lists)

In [21]:
def get_source_name(loc_list):
    try:
        primary = loc_list[0]
        return primary["source"]["display_name"]
    except:
        return None

def get_source_type(loc_list):
    try:
        primary = loc_list[0]
        return primary["source"]["type"]
    except:
        return None

def get_funder_names(funder_list):
    """
    funder_list is a list of grant dicts; return unique funder_display_name values.
    """
    try:
        return list(set([f['funder_display_name'] for f in funder_list]))
    except:
        return []

In [22]:
dftriple["source"] = dftriple["paper_locations"].apply(get_source_name)
dftriple["source_type"] = dftriple["paper_locations"].apply(get_source_type)
dftriple["funder_list"] = dftriple["paper_grants"].apply(get_funder_names)

In [23]:
# full dfinfo (indexed by paper id) with everything jamming2d.py reads
dfinfo = dfpapers[['x','y','id','title','abstract', 'doi','cluster','probability',
                 'publication_date','grants','locations',
                   'keywords','top_concepts']].copy()

dfinfo['affil_list'] = dftriple.groupby('paper_id')['paper_raw_affiliation_string'].\
    apply(lambda x: x.tolist())
dfinfo['author_list'] = dftriple.groupby('paper_id')['paper_author_display_name'].\
    apply(lambda x: x.tolist())

dfinfo['wrapped_affil_list'] = dfinfo['affil_list'].apply(str).apply(wrap_it)
dfinfo['wrapped_author_list'] = dfinfo['author_list'].apply(str).apply(wrap_it)
dfinfo['wrapped_keywords'] = dfinfo['keywords'].apply(str).apply(wrap_it)

dfinfo["source"] = dfinfo["locations"].apply(get_source_name)
dfinfo["source_type"] = dfinfo["locations"].apply(get_source_type)
dfinfo["funder_list"] = dfinfo["grants"].apply(get_funder_names)
dfinfo["wrapped_funder_list"] = dfinfo["funder_list"].apply(str).apply(wrap_it)

## 7. Write the three frames (gzip pickles the app loads)

In [24]:
centroids.to_pickle('jammingcrpacentroids2d.pkl.gz')
dftriple.to_pickle('jammingcrpadftriple2d.pkl.gz')
dfinfo.to_pickle('jammingcrpadfinfo2d.pkl.gz')
print("wrote centroids", centroids.shape, "| dftriple", dftriple.shape, "| dfinfo", dfinfo.shape)

wrote centroids (26, 7) | dftriple (7748, 27) | dfinfo (3018, 22)


## 8. Source homepage dict + affiliation geo dict

In [25]:
sources_list = [s for s in dftriple['source'].unique().tolist() if s]
print("unique sources:", len(sources_list))

def get_source_json(s:str):
    """s is an OpenAlex Sources display_name; return its homepage_url (or None)."""
    source_json = Sources().search_filter(display_name=s).get()
    if source_json and "homepage_url" in source_json[0] and source_json[0]['homepage_url']:
        return source_json[0]["homepage_url"]
    return None

def get_display_page_dict(sl:list):
    mapping_dict = dict()
    for s in tqdm(sl, desc="sources"):
        try:
            mapping_dict[s] = get_source_json(s)
        except:
            pass
    return mapping_dict

source_page_dict = get_display_page_dict(sources_list)

import pickle
with open("updatesource_page_dict.pkl", "wb") as f:
    pickle.dump(source_page_dict, f)

unique sources: 461


sources:   0%|          | 0/461 [00:00<?, ?it/s]

sources:   0%|          | 1/461 [00:01<09:56,  1.30s/it]

sources:   0%|          | 2/461 [00:01<06:10,  1.24it/s]

sources:   1%|          | 3/461 [00:02<05:07,  1.49it/s]

...

sources:  96%|█████████▌| 443/461 [04:10<00:08,  2.19it/s]

sources:  96%|█████████▋| 444/461 [04:10<00:08,  2.12it/s]

sources:  97%|█████████▋| 445/461 [04:11<00:07,  2.16it/s]

sources:  97%|█████████▋| 446/461 [04:18<00:37,  2.50s/it]

sources: 100%|██████████| 461/461 [04:18<00:00,  1.78it/s]

In [26]:
affils_list = [a for a in dftriple['display_name'].unique().tolist() if a]
print("unique affiliations:", len(affils_list))

def get_affil_json(s:str):
    """s is an OpenAlex Institutions display_name; return (lat, lon) or (None, None)."""
    affil_json = Institutions().search_filter(display_name=s).get()
    if affil_json and "geo" in affil_json[0]:
        return affil_json[0]["geo"]["latitude"], affil_json[0]["geo"]["longitude"]
    return None, None

def get_display_geo_dict(sl:list):
    mapping_dict = dict()
    for s in tqdm(sl, desc="affils"):
        try:
            mapping_dict[s] = get_affil_json(s)
        except:
            pass
    return mapping_dict

affil_geo_dict = get_display_geo_dict(affils_list)

with open("updateaffil_geo_dict.pkl", "wb") as f:
    pickle.dump(affil_geo_dict, f)
print("done.")

unique affiliations: 1391


affils:   0%|          | 0/1391 [00:00<?, ?it/s]

affils:   0%|          | 1/1391 [00:00<09:41,  2.39it/s]

affils:   0%|          | 2/1391 [00:00<11:18,  2.05it/s]

affils:   0%|          | 3/1391 [00:01<10:53,  2.12it/s]

...

affils:  95%|█████████▌| 1322/1391 [06:05<00:18,  3.79it/s]

affils:  97%|█████████▋| 1348/1391 [06:11<00:10,  4.27it/s]

affils:  99%|█████████▊| 1373/1391 [06:16<00:03,  4.51it/s]

affils: 100%|██████████| 1391/1391 [06:12<00:00,  3.74it/s]

done.
